# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_books(min_rating=4.0, max_price=20.0):
    """
    Scrape book data from Books to Scrape website based on minimum rating and maximum price.
    
    Parameters:
    min_rating (float): Minimum star rating (1-5) to filter books
    max_price (float): Maximum price in pounds to filter books
    
    Returns:
    pandas.DataFrame: DataFrame containing filtered book data
    """
    
    base_url = "http://books.toscrape.com/"
    all_books = []
    
    # First get all category links
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    category_links = [base_url + li.find('a')['href'] for li in soup.select('.side_categories ul li ul li')]
    
    for category_link in category_links:
        page_url = category_link
        while True:
            # Get page content
            response = requests.get(page_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Get book links on current page
            book_links = [base_url + 'catalogue/' + h3.find('a')['href'].replace('../', '') 
                         for h3 in soup.select('h3')]
            
            for book_link in book_links:
                try:
                    # Get detailed book information
                    book_response = requests.get(book_link)
                    book_soup = BeautifulSoup(book_response.text, 'html.parser')
                    
                    # Extract book details
                    upc = book_soup.find('th', text='UPC').find_next_sibling('td').text
                    title = book_soup.find('h1').text
                    price = float(book_soup.find('th', text='Price (incl. tax)').find_next_sibling('td').text[1:])
                    
                    # Convert rating to numerical value
                    rating_text = book_soup.select_one('.star-rating')['class'][1]
                    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
                    rating = rating_map.get(rating_text, 0)
                    
                    genre = book_soup.find('th', text='Genre').find_next_sibling('td').text
                    availability = book_soup.find('th', text='Availability').find_next_sibling('td').text.strip()
                    description = book_soup.find('meta', attrs={'name': 'description'})['content'].strip()
                    
                    # Check if book meets criteria
                    if rating >= min_rating and price <= max_price:
                        all_books.append({
                            'UPC': upc,
                            'Title': title,
                            'Price (£)': price,
                            'Rating': rating,
                            'Genre': genre,
                            'Availability': availability,
                            'Description': description
                        })
                
                except Exception as e:
                    print(f"Error processing book {book_link}: {str(e)}")
                    continue
            
            # Check for next page
            next_button = soup.select_one('li.next a')
            if next_button:
                page_url = category_link.replace('index.html', '') + next_button['href']
            else:
                break
    
    # Create DataFrame
    df = pd.DataFrame(all_books)
    
    return df




In [ ]:
df=scrape_books(min_rating=4.0,max_price=20.0)
print(df)

C:\Users\shijin kunju\AppData\Local\Temp\ipykernel_14088\2195939666.py:43: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  upc = book_soup.find('th', text='UPC').find_next_sibling('td').text
C:\Users\shijin kunju\AppData\Local\Temp\ipykernel_14088\2195939666.py:45: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  price = float(book_soup.find('th', text='Price (incl. tax)').find_next_sibling('td').text[1:])


Error processing book http://books.toscrape.com/catalogue/its-only-the-himalayas_981/index.html: could not convert string to float: '£45.17'
Error processing book http://books.toscrape.com/catalogue/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html: could not convert string to float: '£49.43'
Error processing book http://books.toscrape.com/catalogue/see-america-a-celebration-of-our-national-parks-treasured-sites_732/index.html: could not convert string to float: '£48.87'
Error processing book http://books.toscrape.com/catalogue/vagabonding-an-uncommon-guide-to-the-art-of-long-term-world-travel_552/index.html: could not convert string to float: '£36.94'
Error processing book http://books.toscrape.com/catalogue/under-the-tuscan-sun_504/index.html: could not convert string to float: '£37.33'
Error processing book http://books.toscrape.com/catalogue/a-summer-in-europe_458/index.html: could not convert string to float: '£44.34'
Error processing book http://books.